# 코스피·코스닥 전 종목 배당성장주 통합 스크리너 (Google Colab + HTML 리포트)

코스피·코스닥 상장 보통주를 대상으로 **기존 1~9번 배당성장주 정량 규칙**과 **선택적 배당 함정 보완 필터**를 한 번에 실행하고, **모바일·PC 최적화 HTML 리포트**로 내려받는 독립 실행 노트북입니다.

### 준비물: OpenDART 무료 API 키 (`DART_API_KEY`) — 1분 설정
1. [OpenDART](https://opendart.fss.or.kr/) 로그인 → 인증키 신청(무료, 즉시 발급)
2. Colab 화면 왼쪽 메뉴의 **🔑(보안 비밀 / Secrets)** 클릭 → **「+ 새 보안 비밀 추가」**
   - 이름: `DART_API_KEY` / 값: 발급받은 키 / **노트북 액세스 켜기**
   - *(보안 비밀을 설정하지 않아도 실행 시 안전한 숨김 입력창으로 바로 입력할 수 있습니다. 키는 코드나 출력에 절대 저장되지 않습니다.)*

### 실행 파이프라인 (약 3~5분 소요, 시총 3,000억 기준)
1. **1단계 (무인증 · 10초):** 네이버 시총 API로 코스피·코스닥 전 종목 수집 → ETF·리츠·스팩·우선주 제외 → 시총 기준 필터
2. **2단계 (OpenDART 1차 · 2~3분):** 3년 단위 사업보고서 4회 호출로 최근 12년 배당·EPS·배당성향 수집 → 1차 배당 이력 통과 종목 압축
3. **3단계 (OpenDART + 네이버 2차 · 1~2분):** 1차 통과 종목에 1~9번 정량 판정 + 선행 컨센서스 밸류에이션 + 52주 낙폭·60일 거래대금 수집
4. **4단계 (리포트 다운로드):** 원칙 부합/상위 후보 표 · 사분면 차트 · **단일 HTML 리포트(`screener_report_YYYY-MM-DD.html`) 및 CSV 자동 다운로드**

In [ ]:
# ==============================================================================
# 1. 환경 설정, 한글 폰트 자동 구성 및 OpenDART API 키 안전 로드
# ==============================================================================
import os, sys, re, io, json, math, time, base64, zipfile, getpass, warnings, statistics
import xml.etree.ElementTree as ET
from pathlib import Path
from datetime import date, datetime
from html import escape
from concurrent.futures import ThreadPoolExecutor, as_completed
import subprocess

warnings.filterwarnings("ignore")
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    print("[환경] Google Colab 감지 -> 한글 폰트(fonts-nanum) 설치 중...")
    subprocess.run(["apt-get", "-qq", "update"], check=False, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    subprocess.run(["apt-get", "-qq", "install", "-y", "fonts-nanum"], check=False, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import matplotlib.ticker as mtick

try:
    from IPython.display import display
except ImportError:
    display = print

def setup_korean_font():
    chosen = None
    for p in [
        "/usr/share/fonts/truetype/nanum/NanumGothic.ttf",
        "/usr/share/fonts/truetype/nanum/NanumBarunGothic.ttf",
    ]:
        if os.path.exists(p):
            fm.fontManager.addfont(p)
            if chosen is None:
                chosen = fm.FontProperties(fname=p).get_name()
    if chosen is None:
        avail = {f.name for f in fm.fontManager.ttflist}
        for cand in ["NanumGothic", "AppleGothic", "Malgun Gothic", "DejaVu Sans"]:
            if cand in avail:
                chosen = cand
                break
    try:
        plt.style.use("seaborn-v0_8-whitegrid")
    except Exception:
        pass
    if chosen:
        plt.rcParams["font.family"] = chosen
    plt.rcParams["axes.unicode_minus"] = False
    plt.rcParams["figure.dpi"] = 120
    return chosen

def load_dart_api_key() -> str:
    key = ""
    if IN_COLAB:
        try:
            from google.colab import userdata
            key = (userdata.get("DART_API_KEY") or "").strip()
        except Exception:
            key = ""
    if not key:
        key = os.environ.get("DART_API_KEY", "").strip()
    if not key:
        key = getpass.getpass("OpenDART API Key(DART_API_KEY)를 입력하세요 (화면 미표시): ").strip()
    if not key:
        raise RuntimeError("DART_API_KEY가 필요합니다. OpenDART(opendart.fss.or.kr)에서 무료 키를 발급받아 입력해 주세요.")
    return key

FONT_NAME = setup_korean_font()
DART_API_KEY = load_dart_api_key()
print(f"[완료] 한글 폰트: {FONT_NAME} | DART_API_KEY 로드 완료 (길이 {len(DART_API_KEY)}자, 마스킹 보호)")

In [ ]:
# ==============================================================================
# 2. 스크리너 파라미터 설정 (자유롭게 수정 후 모두 실행)
# ==============================================================================
SCREENER_CONFIG = {
    # 분석 기준 사업연도 (최근 사업보고서 확정 연도)
    "FISCAL_YEAR": 2025,

    # 탐색 대상 시장 및 최소 시가총액 (단위: 억 원)
    # - 3000: 코스피·코스닥 시총 3,000억 이상 (~570~620종목, 약 3~5분, DART ~3,500회)
    # - 1000: 시총 1,000억 이상 (~1,200종목, 약 6~8분, DART ~6,000회)
    # -    0: 코스피·코스닥 전 종목 (~2,500종목, 약 12~18분, DART ~11,000회)
    "MARKETS": ["KOSPI", "KOSDAQ"],
    "MIN_MCAP_EOK": 3000,

    # 1차 배당 이력 선별 기준
    "MIN_CONSECUTIVE_DIV_YEARS": 10,  # 최근 10년 연속 배당 (원형 1번 PASS 기준)
    "MAX_CUTS_5Y": 2,                 # 최근 5년 DPS 삭감 2회 이하
    "MAX_PAYOUT_3Y": 75.0,            # 3년 합 배당성향 75% 이하

    # 보완 필터 토글 (colab_backtest.ipynb 의 Ablation 결과를 보고 원하는 것만 True로 켜세요!)
    "USE_YIELD_GUARD": False,         # True: 배당수익률 2.0% ~ 12.0% 제한
    "MIN_YIELD": 0.02,
    "MAX_YIELD": 0.12,
    "USE_DPS_CAGR_FLOOR": False,      # True: 정기 DPS 5년 CAGR >= 3.0%
    "MIN_DPS_CAGR_5Y": 0.03,
    "USE_LIQUIDITY_GUARD": False,     # True: 60일 일평균 거래대금 >= 10억 원 (False여도 표에 참고 표시)
    "MIN_TURNOVER_60D": 1_000_000_000,
    "USE_CRASH_EPS_GUARD": False,     # True: 52주 고점 대비 -35% 이하 & 최근 EPS 역성장 동시 제외
    "MAX_DD_52W_WITH_EPS_DROP": -0.35,
    "USE_VAL_CAP": False,             # True: 밸류에이션 개별 지표 괴리율 ±50% 캡 적용
    "VAL_CAP": 0.50,

    # 병렬 작업자 수 및 캐시 경로
    "WORKERS": 6,
    "CACHE_DIR": "./_cache_screener",

    # 국고채 10년물 연평균 금리 CSV 소스 (Common_Everything Public)
    "GITHUB_REPO": "khs0221/Common_Everything",
    "GITHUB_BRANCHES": ["feat/dividend-backtest", "main", "copilot/ai-coding-curriculum"],
    "DATA_SUBPATH": "dividend-backtest/data",
}

CACHE_DIR = Path(SCREENER_CONFIG["CACHE_DIR"])
CACHE_DIR.mkdir(parents=True, exist_ok=True)

active_guards = [k for k in ["USE_YIELD_GUARD", "USE_DPS_CAGR_FLOOR", "USE_LIQUIDITY_GUARD", "USE_CRASH_EPS_GUARD", "USE_VAL_CAP"] if SCREENER_CONFIG[k]]
print(f"[설정] FY{SCREENER_CONFIG['FISCAL_YEAR']} | 대상: {SCREENER_CONFIG['MARKETS']} 시총 {SCREENER_CONFIG['MIN_MCAP_EOK']:,}억 원 이상")
print(f"[설정] 활성화된 보완 필터: {active_guards or ['없음 (현행 규칙 기준선 그대로)']}")

In [ ]:
# ==============================================================================
# 3. 1단계: 코스피·코스닥 전 종목 시총 목록 수집 (무인증) + DART 고유번호 매핑
# ==============================================================================
_HEADERS = {"User-Agent": "Mozilla/5.0 (dividend-screener-colab)"}
_EXCLUDE_RE = re.compile(r"(ETF|ETN|리츠|스팩|SPAC|인프라|선박투자|펀드|KODEX|TIGER|ACE|KBSTAR|SOL|ARIRANG|HANARO|PLUS|RISE)")
_PATH_HINT = [
    ("securities", r"증권"),
    ("financial_holding", r"캐피탈|카드|투자|한국금융지주|메리츠금융"),
    ("bank", r"은행|금융지주|신한지주|우리금융|하나금융|BNK|DGB|iM금융|JB금융|KB금융|금융$"),
    ("insurance", r"생명(?!과학|공학|자원)|화재|보험|손보|코리안리"),
]

def guess_path(name: str) -> str:
    for p, pat in _PATH_HINT:
        if re.search(pat, str(name)):
            return p
    return "general"

def _num(v):
    if v in (None, "", "-", "－", "N/A"):
        return None
    s = str(v).replace(",", "").replace("배", "").replace("원", "").replace("%", "").strip()
    try:
        x = float(s)
        return None if (math.isnan(x) or math.isinf(x)) else x
    except ValueError:
        return None

def _num_dash0(v):
    return 0.0 if str(v).strip() in ("-", "－") else _num(v)

def fetch_naver_market_list(market: str) -> pd.DataFrame:
    rows, page = [], 1
    while True:
        r = requests.get(
            f"https://m.stock.naver.com/api/stocks/marketValue/{market}",
            params={"page": page, "pageSize": 100}, headers=_HEADERS, timeout=20
        )
        r.raise_for_status()
        stocks = r.json().get("stocks") or []
        if not stocks:
            break
        for s in stocks:
            rows.append({
                "ticker": str(s["itemCode"]).zfill(6),
                "name": s["stockName"],
                "close": _num(s.get("closePrice")),
                "mcap_eok": _num(s.get("marketValue")),
                "end_type": s.get("stockEndType"),
                "market": market,
            })
        page += 1
        time.sleep(0.1)
    df = pd.DataFrame(rows)
    df["is_pref"] = df["ticker"].str[-1] != "0"
    df["excluded"] = (df["end_type"] != "stock") | df["name"].str.contains(_EXCLUDE_RE)
    return df

def fetch_corp_codes() -> dict[str, str]:
    cache_p = CACHE_DIR / "corp_codes.json"
    if cache_p.exists():
        return json.loads(cache_p.read_text(encoding="utf-8"))
    r = requests.get("https://opendart.fss.or.kr/api/corpCode.xml", params={"crtfc_key": DART_API_KEY}, timeout=90)
    if r.status_code != 200:
        raise RuntimeError(f"OpenDART corpCode.xml HTTP {r.status_code}")
    with zipfile.ZipFile(io.BytesIO(r.content)) as z:
        xml_bytes = z.read(z.namelist()[0])
    root = ET.fromstring(xml_bytes)
    mapping = {}
    for el in root.iter("list"):
        sc = (el.findtext("stock_code") or "").strip()
        cc = (el.findtext("corp_code") or "").strip()
        if sc and cc:
            mapping[sc.zfill(6)] = cc
    cache_p.write_text(json.dumps(mapping), encoding="utf-8")
    return mapping

def load_rf_rates() -> tuple[dict[int, float], float]:
    # 연도별 국고채 10년물 연평균 금리 (Common_Everything Public CSV)
    rf_map = {2015:0.0230, 2016:0.0175, 2017:0.0228, 2018:0.0250, 2019:0.0170,
              2020:0.0150, 2021:0.0207, 2022:0.0337, 2023:0.0364, 2024:0.0322, 2025:0.0289}
    for br in SCREENER_CONFIG["GITHUB_BRANCHES"]:
        url = f"https://raw.githubusercontent.com/{SCREENER_CONFIG['GITHUB_REPO']}/{br}/{SCREENER_CONFIG['DATA_SUBPATH']}/kr_treasury_10y_annual.csv"
        try:
            r = requests.get(url, timeout=15)
            if r.status_code == 200:
                df = pd.read_csv(io.StringIO(r.text))
                rf_map = {int(row["연도"]): float(row["국고채10년_연평균_퍼센트"]) / 100.0 for _, row in df.iterrows()}
                break
        except Exception:
            pass
    # 현재 국고채 10년물 금리 (네이버)
    cur_rf = rf_map[max(rf_map)]
    try:
        rj = requests.get("https://api.stock.naver.com/marketindex/bond/KR10YT=RR", headers=_HEADERS, timeout=10).json()
        cur_rf = float(rj["closePrice"]) / 100.0
    except Exception:
        pass
    return rf_map, cur_rf

print("[1단계] 코스피·코스닥 전 종목 목록 및 DART 고유번호 수집 중...")
all_mkt = pd.concat([fetch_naver_market_list(m) for m in SCREENER_CONFIG["MARKETS"]], ignore_index=True).drop_duplicates("ticker")
CORP_MAP = fetch_corp_codes()
RF_MAP, CUR_RF = load_rf_rates()

uni = all_mkt[(~all_mkt["excluded"]) & (~all_mkt["is_pref"]) & (all_mkt["mcap_eok"].fillna(0) >= SCREENER_CONFIG["MIN_MCAP_EOK"])].copy()
uni["corp_code"] = uni["ticker"].map(CORP_MAP)
uni = uni[uni["corp_code"].notna()].reset_index(drop=True)
uni["path"] = uni["name"].map(guess_path)

print(f"[완료] 전체 상장 {len(all_mkt):,}종목 -> 보통주·시총 {SCREENER_CONFIG['MIN_MCAP_EOK']:,}억↑ & DART 매핑 완료: {len(uni):,}종목")
print(f"       시장별: {uni['market'].value_counts().to_dict()} | 국고채 10년물 현재 금리: {CUR_RF:.2%}")

In [ ]:
# ==============================================================================
# 4. 2단계: OpenDART 배당 이력 고속 1차 선별 (3년 단위 4회 호출 -> 12년치 확보)
# ==============================================================================
DART_BASE = "https://opendart.fss.or.kr/api"
DART_CALLS = {"count": 0, "limit_hit": False}

def dart_get(endpoint: str, corp_code: str, bsns_year: int, extra: dict | None = None) -> list[dict]:
    if DART_CALLS["limit_hit"]:
        return []
    key_tag = f"{endpoint}_{corp_code}_{bsns_year}" + (f"_{extra.get('fs_div','')}" if extra else "")
    cache_p = CACHE_DIR / f"{key_tag}.json"
    if cache_p.exists():
        return json.loads(cache_p.read_text(encoding="utf-8"))

    params = {"crtfc_key": DART_API_KEY, "corp_code": corp_code, "bsns_year": str(bsns_year), "reprt_code": "11011"}
    if extra:
        params.update(extra)

    for attempt in range(4):
        try:
            DART_CALLS["count"] += 1
            r = requests.get(f"{DART_BASE}/{endpoint}.json", params=params, timeout=30)
            if r.status_code != 200:
                time.sleep(1.5 * (attempt + 1))
                continue
            j = r.json()
            st = j.get("status")
            if st == "020":   # 일일 요청 제한 초과
                DART_CALLS["limit_hit"] = True
                print("[경고] OpenDART 일일 요청 한도(020)에 도달했습니다. 지금까지 수집된 캐시로 진행합니다.")
                return []
            if st == "013":   # 데이터 없음
                cache_p.write_text("[]", encoding="utf-8")
                return []
            if st == "000":
                rows = j.get("list") or []
                cache_p.write_text(json.dumps(rows, ensure_ascii=False), encoding="utf-8")
                return rows
            return []
        except Exception:
            time.sleep(1.0 * (attempt + 1))
    return []

def _stock_class(s: str) -> str | None:
    s = str(s or "")
    if "보통" in s or "의결권 있는" in s or "의결권있는" in s:
        return "common"
    if "우선" in s or "종류주" in s or "의결권 없는" in s or "의결권없는" in s:
        return "pref"
    return None

def parse_alot_12y(corp_code: str, last_year: int) -> dict[int, dict]:
    '''last_year, -3, -6, -9 총 4회 호출로 최근 12개 사업연도 배당·EPS·순이익·배당성향 추출.'''
    by_year: dict[int, dict] = {}
    call_years = [y for y in (last_year, last_year - 3, last_year - 6, last_year - 9) if y >= 2015]
    for cy in call_years:
        rows = dart_get("alotMatter", corp_code, cy)
        if not rows:
            continue
        # 결산일이 여러 개면 가장 늦은 결산일만 사용
        st_dates = sorted({str(r.get("stlm_dt") or "").strip() for r in rows if str(r.get("stlm_dt") or "").strip() not in ("", "-")})
        if len(st_dates) > 1:
            rows = [r for r in rows if str(r.get("stlm_dt") or "").strip() == st_dates[-1]]
        cfs_ni = any("당기순이익" in str(r.get("se", "")) and "연결" in str(r.get("se", "")) for r in rows)

        for col, lag in (("thstrm", 0), ("frmtrm", 1), ("lwfr", 2)):
            y = cy - lag
            if y in by_year and by_year[y].get("dps_common") is not None:
                continue
            out = by_year.setdefault(y, {})
            period_sum, total_dash0 = 0.0, False
            for r in rows:
                se = str(r.get("se", ""))
                knd_raw = str(r.get("stock_knd") or "").strip()
                knd = "common" if knd_raw in ("", "-", "nan", "None") else _stock_class(knd_raw)
                v = _num(r.get(col))
                if "주당 현금배당금" in se and re.search(r"결산|중간|분기", knd_raw):
                    if v is not None:
                        period_sum += v
                    continue
                if "액면" in se and v is not None:
                    out["face_value"] = v
                elif "당기순이익" in se and "별도" not in se and not ("개별" in se and cfs_ni):
                    if v is not None: out["ni_reported"] = v * 1e6
                elif "주당순이익" in se and v is not None:
                    out["eps_reported"] = v
                elif "현금배당금총액" in se:
                    if v is not None: out["div_total_reported"] = v * 1e6
                    total_dash0 = _num_dash0(r.get(col)) == 0.0
                elif "현금배당성향" in se and v is not None:
                    out["payout_reported"] = v
                elif "주당 현금배당금" in se and knd == "common" and out.get("dps_common") is None:
                    out["dps_common"] = v
            if out.get("dps_common") is None and period_sum > 0:
                out["dps_common"] = period_sum
            if out.get("dps_common") is None and out.get("eps_reported") is not None and total_dash0:
                out["dps_common"] = 0.0
                out["payout_reported"] = 0.0

    # 액면분할 보정 (최신 액면가 기준)
    latest_fv = next((by_year[y]["face_value"] for y in sorted(by_year, reverse=True) if by_year[y].get("face_value")), None)
    if latest_fv and latest_fv > 0:
        for y, d in by_year.items():
            fv = d.get("face_value")
            if fv and fv > 0 and abs(fv - latest_fv) > 0.5:
                ratio = latest_fv / fv
                for k in ("dps_common", "eps_reported"):
                    if d.get(k) is not None:
                        d[k] = d[k] * ratio
    return by_year

def avg3_dict(by_year: dict, col: str, end: int):
    vals = [by_year[y][col] for y in (end - 2, end - 1, end) if y in by_year and by_year[y].get(col) is not None]
    return sum(vals) / len(vals) if len(vals) >= 2 else None

def cagr3_dict(by_year: dict, col: str, end: int, span: int):
    a, b = avg3_dict(by_year, col, end - span), avg3_dict(by_year, col, end)
    if a is None or a <= 0 or b is None:
        return None
    return -1.0 if b <= 0 else (b / a) ** (1.0 / span) - 1.0

def stage1_worker(row_dict: dict, last_year: int) -> dict:
    corp = row_dict["corp_code"]
    by_year = parse_alot_12y(corp, last_year)
    min_yrs = SCREENER_CONFIG["MIN_CONSECUTIVE_DIV_YEARS"]
    paid_run = 0
    for y in range(last_year, last_year - 13, -1):
        if (by_year.get(y) or {}).get("dps_common") and by_year[y]["dps_common"] > 0:
            paid_run += 1
        else:
            break
    g10 = cagr3_dict(by_year, "dps_common", last_year, 10)
    g5 = cagr3_dict(by_year, "dps_common", last_year, 5)
    e5 = cagr3_dict(by_year, "eps_reported", last_year, 5)
    cuts5 = 0
    for y in range(last_year - 4, last_year + 1):
        a = (by_year.get(y - 1) or {}).get("dps_common")
        b = (by_year.get(y) or {}).get("dps_common")
        if a is not None and b is not None and b < a - min(0.5, abs(a) * 1e-6):
            cuts5 += 1
    # 3년 합 배당성향
    d3 = [by_year.get(y, {}).get("div_total_reported") for y in (last_year - 2, last_year - 1, last_year)]
    n3 = [by_year.get(y, {}).get("ni_reported") for y in (last_year - 2, last_year - 1, last_year)]
    p3 = (sum(d3) / sum(n3) * 100.0) if (all(v is not None for v in d3 + n3) and sum(n3) > 0) else None

    pass1 = (
        paid_run >= min_yrs
        and g5 is not None and g5 > 0
        and (g10 is None or g10 > 0)
        and (e5 is None or e5 >= 0)
        and cuts5 <= SCREENER_CONFIG["MAX_CUTS_5Y"]
        and p3 is not None and 0 < p3 <= SCREENER_CONFIG["MAX_PAYOUT_3Y"]
    )
    return {
        **row_dict, "pass1": pass1, "paid_run": paid_run,
        "dps_g10": g10, "dps_g5": g5, "eps_g5": e5, "cuts_5y": cuts5, "payout3_div": p3,
        "dps_last": (by_year.get(last_year) or {}).get("dps_common"),
        "by_year_div": by_year,
    }

FY = SCREENER_CONFIG["FISCAL_YEAR"]
print(f"[2단계] {len(uni):,}개 종목 OpenDART 1차 배당 이력 스캔 시작 (FY{FY})...")
t0 = time.time()
stage1_results = []
with ThreadPoolExecutor(max_workers=SCREENER_CONFIG["WORKERS"]) as ex:
    futs = [ex.submit(stage1_worker, row.to_dict(), FY) for _, row in uni.iterrows()]
    for idx, fut in enumerate(as_completed(futs), 1):
        stage1_results.append(fut.result())
        if idx % 100 == 0 or idx == len(futs):
            n_ok = sum(1 for x in stage1_results if x["pass1"])
            print(f"  진행 {idx:,}/{len(futs):,} | 1차 통과 {n_ok}종목 | DART 호출 {DART_CALLS['count']:,}회 ({time.time()-t0:.0f}초)")

SURVIVORS = [x for x in stage1_results if x["pass1"]]
print(f"[완료] 1차 배당 이력 통과: {len(SURVIVORS)}종목 / {len(uni):,}종목 ({time.time()-t0:.1f}초)")

In [ ]:
# ==============================================================================
# 5. 3단계: 1차 통과 종목 정밀 수집 (DART 재무·주식·자사주 + 네이버 시세·컨센서스)
# ==============================================================================
_FCHART_URL = "https://fchart.stock.naver.com/sise.nhn"
_ITEM_RE = re.compile(r'<item data="([^"]+)"')

def fetch_naver_daily(ticker: str, count: int = 3200) -> pd.DataFrame:
    for attempt in range(3):
        try:
            r = requests.get(_FCHART_URL, params={"symbol": ticker, "timeframe": "day", "count": count, "requestType": 0}, headers=_HEADERS, timeout=20)
            r.raise_for_status()
            rows = [(pd.Timestamp(d), float(o), float(h), float(l), float(c), float(v))
                    for d, o, h, l, c, v in (m.group(1).split("|") for m in _ITEM_RE.finditer(r.content.decode("euc-kr", errors="ignore")))]
            df = pd.DataFrame(rows, columns=["date", "open", "high", "low", "close", "volume"]).set_index("date")
            df = df[df["close"] > 0].sort_index()
            if not df.empty:
                return df
        except Exception:
            time.sleep(0.4 * (attempt + 1))
    return pd.DataFrame()

def fetch_naver_consensus(ticker: str, target_fy: int) -> dict | None:
    try:
        r = requests.get(f"https://m.stock.naver.com/api/stock/{ticker}/finance/annual", headers=_HEADERS, timeout=15)
        info = r.json().get("financeInfo") or {}
        cons_cols = [t for t in info.get("trTitleList", []) if t.get("isConsensus") == "Y"]
        if not cons_cols:
            return None
        col = cons_cols[0]
        m = re.match(r"(\d{4})", str(col.get("title") or ""))
        if m and int(m.group(1)) != target_fy:
            return None
        key = col["key"]
        out = {"year": col["title"]}
        name_map = {"EPS": "eps", "BPS": "bps", "주당배당금": "dps"}
        for row in info.get("rowList", []):
            k = name_map.get(row.get("title"))
            if k:
                out[k] = _num(((row.get("columns") or {}).get(key) or {}).get("value"))
        return out if (out.get("eps") and out["eps"] > 0) else None
    except Exception:
        return None

def extract_fs_year(rows: list[dict], col: str) -> dict:
    '''OpenDART fnlttSinglAcntAll 응답에서 핵심 재무 계정 추출.'''
    def find(ids, nm_pat, sj_pref=None):
        cand = rows
        if sj_pref:
            sub = [r for r in rows if r.get("sj_div") in sj_pref]
            if sub:
                cand = sub
        for aid in ids:
            for r in cand:
                rid = re.sub(r"^(ifrs-full|ifrs|dart)_", "", str(r.get("account_id") or ""))
                if rid == aid:
                    v = _num(r.get(col))
                    if v is not None:
                        return v
        if nm_pat:
            rx = re.compile(nm_pat)
            for r in cand:
                if rx.match(str(r.get("account_nm") or "").strip()):
                    v = _num(r.get(col))
                    if v is not None:
                        return v
        return None

    op = find(["OperatingIncomeLoss"], r"^영업(이익|손실|순손익|이익\(손실\))$", ("IS", "CIS"))
    ni_p = find(["ProfitLossAttributableToOwnersOfParent"], r"^지배(기업|주주)(의)?\s*(소유주|지분)?(?!.*(포괄|주당|중단|계속)).*(당기)?순(이익|손익|손실)", ("IS", "CIS"))
    ni = find(["ProfitLoss"], r"^당기순이익(\(손실\))?$", ("IS", "CIS"))
    cfo = find(["CashFlowsFromUsedInOperatingActivities"], r"^영업활동.*현금흐름", ("CF",))
    capex_ppe = find(["PurchaseOfPropertyPlantAndEquipmentClassifiedAsInvestingActivities", "PurchaseOfPropertyPlantAndEquipment"], r".*유형자산의?\s*(취득|증가)$", ("CF",))
    capex_int = find(["PurchaseOfIntangibleAssetsClassifiedAsInvestingActivities"], r".*무형자산의?\s*취득$", ("CF",))
    tp = find(["PaymentsForPurchaseOfTreasuryShares", "AcquisitionOfTreasuryShares", "PurchaseOfTreasuryShares"], r".*(자기주식|자사주)(의)?\s*(취득|매입)", ("CF",))
    eq_p = find(["EquityAttributableToOwnersOfParent"], r"지배(기업|주주).*지분$", ("BS",))
    eq = find(["Equity"], r"^자\s*본\s*총\s*계$", ("BS",))
    cash = find(["CashAndCashEquivalents", "CashEquivalents"], r"^현금\s*및\s*현금성\s*자산$|^현금성자산$", ("BS",))
    st_fin = find(["ShorttermDepositsNotClassifiedAsCashEquivalents"], r"^단기금융상품$", ("BS",))
    inv_aff = find(["InvestmentsInSubsidiariesJointVenturesAndAssociates", "InvestmentAccountedForUsingEquityMethod"], r"^(종속기업|관계기업|공동기업).*투자", ("BS",))
    int_exp = find(["InterestExpense"], r"^이자비용$", ("IS", "CIS"))
    int_paid = find(["InterestPaidClassifiedAsOperatingActivities"], r"^이자의?\s*지급", ("CF",))
    dep = find(["DepreciationAndAmortisationExpense", "AdjustmentsForDepreciationAndAmortisationExpense"], r"^감가상각비", ("CF", "IS", "CIS"))

    # 차입금·사채 합산 (재무상태표)
    debt_exc = re.compile(r"채권|자산|투자|대여|할인|할증|발행비|이자$|미지급|조정|충당|보증|리스|파생")
    debt_seen, debt_sum, debt_found = set(), 0.0, False
    for r in rows:
        if r.get("sj_div") != "BS":
            continue
        nm = str(r.get("account_nm") or "").replace(" ", "")
        if ("차입" in nm or "사채" in nm or nm.startswith("유동성장기부채")) and not debt_exc.search(nm):
            v = _num(r.get(col))
            if v is not None and (nm, v) not in debt_seen:
                debt_seen.add((nm, v))
                debt_sum += abs(v)
                debt_found = True
    debt = debt_sum if debt_found else (0.0 if (cash is not None and op and abs(int_exp or int_paid or 0) < abs(op) * 0.05) else None)

    capex = (abs(capex_ppe) + abs(capex_int or 0.0)) if capex_ppe is not None else None
    fcf = (cfo - capex) if (cfo is not None and capex is not None) else None
    net_debt = (debt - cash - (st_fin or 0.0)) if (debt is not None and cash is not None) else None
    ebitda = (op + abs(dep)) if (op is not None and dep is not None) else None
    interest = int_exp if int_exp else int_paid
    icr = (op / abs(interest)) if (op is not None and interest and abs(interest) > 0) else None
    nde = (net_debt / ebitda) if (net_debt is not None and ebitda is not None and ebitda > 0) else None
    invested = (eq + net_debt - (inv_aff or 0.0)) if (eq is not None and net_debt is not None) else None
    roic = (op * (1 - 0.24) / invested) if (op is not None and invested is not None and invested > 0) else None
    roe_ni = ni_p if ni_p is not None else ni
    roe_eq = eq_p if (ni_p is not None and eq_p) else eq
    roe = (roe_ni / roe_eq) if (roe_ni is not None and roe_eq and roe_eq > 0) else None

    return {
        "op": op, "ni_p": ni_p, "ni": ni, "fcf": fcf, "tp": abs(tp) if tp is not None else 0.0,
        "eq_p": eq_p, "eq": eq, "debt": debt, "net_debt": net_debt, "ebitda": ebitda,
        "icr": icr, "nde": nde, "roic": roic, "roe": roe, "int": abs(interest) if interest else None,
        "has_fs": (ni is not None or eq is not None),
    }

def collect_stage2_one(item: dict, fy: int) -> dict:
    corp, ticker = item["corp_code"], item["ticker"]
    # 1) 재무제표 (CFS 우선, 없으면 OFS) — 3년 단위 4회 호출
    fs_by_year = {}
    for cy in [y for y in (fy, fy - 3, fy - 6, fy - 9) if y >= 2015]:
        rows = dart_get("fnlttSinglAcntAll", corp, cy, {"fs_div": "CFS"})
        if not rows:
            rows = dart_get("fnlttSinglAcntAll", corp, cy, {"fs_div": "OFS"})
        if not rows:
            continue
        for col, lag in (("thstrm_amount", 0), ("frmtrm_amount", 1), ("bfefrmtrm_amount", 2)):
            y = cy - lag
            if y not in fs_by_year or not fs_by_year[y].get("has_fs"):
                ext = extract_fs_year(rows, col)
                if ext["has_fs"]:
                    fs_by_year[y] = ext

    # 2) 주식 총수 (FY 및 FY-5)
    sh_by_year = {}
    for y in (fy, fy - 5):
        rows = dart_get("stockTotqySttus", corp, y)
        if not rows:
            continue
        rcp = max(str(r.get("rcept_no") or "") for r in rows)
        rows = [r for r in rows if str(r.get("rcept_no") or "") == rcp]
        oc, op_sh = 0.0, 0.0
        for r in rows:
            k = _stock_class(r.get("se", ""))
            v = _num(r.get("distb_stock_co"))
            if k == "common" and v: oc += v
            elif k == "pref" and v: op_sh += v
        if oc > 0:
            sh_by_year[y] = {"common": oc, "pref": op_sh}

    # 3) 자기주식 취득·소각 (최근 5년)
    ts_by_year = {}
    for y in range(fy - 4, fy + 1):
        rows = dart_get("tesstkAcqsDspsSttus", corp, y)
        if not rows:
            continue
        acq, canc = 0.0, 0.0
        for r in rows:
            if _stock_class(r.get("stock_knd", "")) == "common" and str(r.get("acqs_mth1")) == "총계":
                acq += _num_dash0(r.get("change_qy_acqs")) or 0.0
                canc += _num_dash0(r.get("change_qy_incnr")) or 0.0
        ts_by_year[y] = {"acq": acq, "canc": canc}

    # 4) 네이버 일별 주가 + 컨센서스
    px_df = fetch_naver_daily(ticker)
    cons = fetch_naver_consensus(ticker, fy + 1)

    return {**item, "fs": fs_by_year, "shares": sh_by_year, "ts": ts_by_year, "px_df": px_df, "cons": cons}

print(f"[3단계] 1차 통과 {len(SURVIVORS)}종목 정밀 수집 중...")
t0 = time.time()
STAGE2_DATA = []
with ThreadPoolExecutor(max_workers=SCREENER_CONFIG["WORKERS"]) as ex:
    futs = [ex.submit(collect_stage2_one, it, FY) for it in SURVIVORS]
    for idx, fut in enumerate(as_completed(futs), 1):
        STAGE2_DATA.append(fut.result())
        if idx % 20 == 0 or idx == len(futs):
            print(f"  진행 {idx}/{len(futs)} | 누적 DART 호출 {DART_CALLS['count']:,}회 ({time.time()-t0:.0f}초)")

print(f"[완료] 2차 정밀 데이터 수집 완료 ({time.time()-t0:.1f}초, 총 DART 호출 {DART_CALLS['count']:,}회)")

In [ ]:
# ==============================================================================
# 6. 4단계: 1~9번 정량 판정 + 재투자 우선순위 + 선택적 보완 필터 적용
# ==============================================================================
ERP, TAX = 0.05, 0.24

def judge_screener_one(d: dict, fy: int) -> dict | None:
    px_df = d["px_df"]
    if px_df is None or px_df.empty:
        return None
    cur_close = float(px_df["close"].iloc[-1])
    turnover_60d = float((px_df["close"] * px_df["volume"]).tail(60).mean())
    w52 = px_df.loc[px_df.index >= px_df.index[-1] - pd.Timedelta(days=365)]
    dd_52w = (cur_close / float(w52["high"].max()) - 1.0) if not w52.empty else 0.0
    ye_px = px_df["close"].groupby(px_df.index.year).last().to_dict()

    path = d["path"]
    div_y, fs_y, sh_y, ts_y = d["by_year_div"], d["fs"], d["shares"], d["ts"]

    # 1번 배당 이력
    v1 = "FAIL" if (d["paid_run"] < 7 or d["cuts_5y"] >= 2) else (
        "PASS" if (d["paid_run"] >= 10 and d["cuts_5y"] == 0 and (d["dps_g5"] or 0) > 0 and (d["dps_g10"] is None or d["dps_g10"] > 0)) else "WARN"
    )

    # 3년 합 총환원율 & 배당성향
    W3 = (fy - 2, fy - 1, fy)
    d3 = [div_y.get(y, {}).get("div_total_reported") for y in W3]
    e3 = [(fs_y.get(y, {}).get("ni_p") if fs_y.get(y, {}).get("ni_p") is not None else div_y.get(y, {}).get("ni_reported")) for y in W3]
    if all(v is not None for v in d3 + e3) and sum(e3) > 0:
        b3 = sum(fs_y.get(y, {}).get("tp", 0.0) for y in W3)
        c3 = sum(ts_y.get(y, {}).get("canc", 0.0) for y in W3)
        a3 = sum(ts_y.get(y, {}).get("acq", 0.0) for y in W3)
        s_rat = min(1.0, c3 / a3) if a3 > 0 else (1.0 if c3 > 0 else 0.0)
        ret3_num = sum(d3) + b3 * s_rat
        tr_3y = ret3_num / sum(e3) * 100.0
        p3 = sum(d3) / sum(e3) * 100.0
    else:
        ret3_num, tr_3y, p3 = None, None, d["payout3_div"]

    # 2번 EPS 성장
    g_eps, g_dps = d["eps_g5"], d["dps_g5"]
    if g_eps is None or g_dps is None: v2 = "NA"
    elif g_eps < 0 and g_dps > 0: v2 = "FAIL"
    elif g_eps > 0 and (g_dps <= g_eps + 0.03 or (p3 is not None and p3 < 30.0)): v2 = "PASS"
    else: v2 = "WARN"

    # 3번 FCF / 순이익 꾸준함
    vals3 = [fs_y.get(y, {}).get("fcf") if path == "general" else (fs_y.get(y, {}).get("ni_p") or div_y.get(y, {}).get("ni_reported"))
             for y in range(fy - 9, fy + 1)]
    k3 = [v for v in vals3 if v is not None]
    if sum(1 for v in k3 if v < 0) >= 3: v3 = "FAIL"
    elif len(k3) >= 5 and sum(1 for v in k3 if v > 0) / len(k3) >= 0.8: v3 = "PASS"
    elif len(k3) >= 5: v3 = "WARN"
    else: v3 = "NA"

    # 4번 ROIC-WACC / ROE-COE
    spreads = []
    for y in range(fy - 4, fy + 1):
        coe = RF_MAP.get(y, CUR_RF) + ERP
        f = fs_y.get(y) or {}
        if path == "general":
            if f.get("roic") is None: continue
            wacc = coe
            sh_tot = (sh_y.get(fy, {}).get("common", 0) + sh_y.get(fy, {}).get("pref", 0))
            mcap_y = ye_px.get(y, 0) * sh_tot
            debt_y = f.get("debt") or 0.0
            if mcap_y > 0:
                kd = (min(0.15, f["int"] / debt_y) if (debt_y > 0 and f.get("int")) else 0.04) * (1 - TAX)
                wacc = coe * (mcap_y / (mcap_y + debt_y)) + kd * (debt_y / (mcap_y + debt_y))
            spreads.append((y, f["roic"] - wacc))
        else:
            if f.get("roe") is not None: spreads.append((y, f["roe"] - coe))
    p_thr, f_thr = (0.03, 0.0) if path == "general" else (0.01, -0.01)
    avg5_sp = sum(s for _, s in spreads) / len(spreads) if len(spreads) >= 3 else None
    l3_sp = [s for y, s in spreads if y >= fy - 2]
    if len(l3_sp) == 3 and all(s < f_thr for s in l3_sp): v4 = "FAIL"
    elif avg5_sp is not None:
        r2 = [s for y, s in spreads if y >= fy - 1]
        v4 = "PASS" if (avg5_sp >= p_thr and (not r2 or sum(r2)/len(r2) >= avg5_sp - 0.03)) else "WARN"
    else: v4 = "NA"

    # 5번 주주환원 수준
    if tr_3y is None or p3 is None: v5 = "NA"
    elif tr_3y < 20.0 or p3 > 75.0 or (path != "general" and tr_3y > 100.0): v5 = "FAIL"
    elif tr_3y >= 30.0 and p3 <= 60.0 and (path == "general" or tr_3y <= 66.7): v5 = "PASS"
    else: v5 = "WARN"

    # 6번 자사주·희석
    s_now = (sh_y.get(fy) or {}).get("common")
    s_old = (sh_y.get(fy - 5) or {}).get("common")
    if not s_now or not s_old: v6 = "NA"
    else:
        chg = s_now / s_old - 1.0
        canc5 = sum(ts_y.get(y, {}).get("canc", 0.0) for y in range(fy - 4, fy + 1))
        acq5 = sum(ts_y.get(y, {}).get("acq", 0.0) for y in range(fy - 4, fy + 1))
        if chg >= 0.03: v6 = "FAIL"
        elif (chg < -0.01 and canc5 > 0) or (-0.01 <= chg <= 0.01 and (acq5 == 0 or canc5 >= acq5)): v6 = "PASS"
        else: v6 = "WARN"

    # 7번 환원 감당 (일반만 판정)
    if path != "general": v7 = "참고"
    else:
        fcfs = [fs_y.get(y, {}).get("fcf") for y in W3]
        if ret3_num is None or any(v is None for v in fcfs): v7 = "NA"
        elif sum(fcfs) <= 0: v7 = "FAIL" if ret3_num > 0 else "PASS"
        else:
            cov = ret3_num / sum(fcfs)
            v7 = "FAIL" if cov > 1.0 else ("WARN" if cov > 2/3 else "PASS")

    # 8번 재무건전성 (금융은 사업보고서 본문 규제자본비율 별도 확인 필요)
    if path != "general":
        v8 = "NA(금융본문)"
    else:
        f_end = fs_y.get(fy) or {}
        nd, nde, icr, eb = f_end.get("net_debt"), f_end.get("nde"), f_end.get("icr"), f_end.get("ebitda")
        if nd is not None and nd <= 0: v8 = "WARN" if (eb is not None and eb <= 0) else "PASS"
        elif (icr is not None and icr < 3.0) or (nde is not None and nde > 3.0): v8 = "FAIL"
        elif nde is not None and icr is not None: v8 = "PASS" if (nde < 2.0 and icr >= 5.0) else "WARN"
        else: v8 = "NA"

    counted = [v1, v2, v3, v4, v5, v6] + ([v7, v8] if path == "general" else [])
    n_fail = counted.count("FAIL")
    n_warn = counted.count("WARN")
    n_na = sum(1 for v in counted if str(v).startswith("NA"))
    if n_fail >= 1 or n_warn >= 5:
        overall = "미부합"
    elif n_warn <= 2 and n_na == 0:
        overall = "원칙 부합" if path == "general" else "원칙 부합(금융자본확인)"
    elif n_na >= 3:
        overall = "판정 보류"
    else:
        overall = "부분 부합"

    # 9번 밸류에이션 (선행 컨센서스 우선 -> FY 기준)
    cons = d["cons"]
    eps_fy = (div_y.get(fy) or {}).get("eps_reported")
    dps_fy = d["dps_last"]
    eq_fy = (fs_y.get(fy) or {}).get("eq_p") or (fs_y.get(fy) or {}).get("eq")
    sh_tot_fy = (sh_y.get(fy, {}).get("common", 0) + sh_y.get(fy, {}).get("pref", 0))
    bps_fy = (eq_fy / sh_tot_fy) if (eq_fy and sh_tot_fy > 0) else None

    if cons:
        val_basis = f"선행({cons['year']})"
        c_per = cur_close / cons["eps"] if cons.get("eps") and cons["eps"] > 0 else None
        c_pbr = cur_close / cons["bps"] if cons.get("bps") and cons["bps"] > 0 else None
        c_yld = cons["dps"] / cur_close if cons.get("dps") and cons["dps"] > 0 else None
    else:
        val_basis = f"FY{fy}"
        c_per = cur_close / eps_fy if (eps_fy and eps_fy > 0) else None
        c_pbr = cur_close / bps_fy if (bps_fy and bps_fy > 0) else None
        c_yld = dps_fy / cur_close if (dps_fy and dps_fy > 0) else None

    h_per, h_pbr, h_yld = [], [], []
    for y in range(fy - 9, fy + 1):
        px = ye_px.get(y)
        if not px or px <= 0: continue
        e = (div_y.get(y) or {}).get("eps_reported")
        dp = (div_y.get(y) or {}).get("dps_common")
        eq = (fs_y.get(y) or {}).get("eq_p") or (fs_y.get(y) or {}).get("eq")
        b = (eq / sh_tot_fy) if (eq and sh_tot_fy > 0) else None
        if e and e > 0: h_per.append(px / e)
        if b and b > 0: h_pbr.append(px / b)
        if dp and dp > 0: h_yld.append(dp / px)

    devs_raw, devs_cap, cheap, dear = [], [], 0, 0
    cap_lim = SCREENER_CONFIG["VAL_CAP"]
    for cv, hv, hc in ((c_per, h_per, False), (c_pbr, h_pbr, False), (c_yld, h_yld, True)):
        if cv is not None and len(hv) >= 5:
            med = statistics.median(hv)
            if med > 0:
                dev = (cv / med - 1.0) if hc else -(cv / med - 1.0)
                devs_raw.append(dev)
                devs_cap.append(max(-cap_lim, min(cap_lim, dev)))
                if dev >= 0: cheap += 1
                if dev < -0.20: dear += 1

    v9 = "NA" if len(devs_raw) < 2 else ("PASS" if cheap >= 2 else ("WARN" if dear >= 2 else "중간"))
    val_raw = sum(devs_raw) / len(devs_raw) if len(devs_raw) >= 2 else None
    val_cap = sum(devs_cap) / len(devs_cap) if len(devs_cap) >= 2 else None
    cur_yld_fy = dps_fy / cur_close if (dps_fy and cur_close > 0) else None

    # 선택적 보완 필터 판정
    eps_prev = (div_y.get(fy - 1) or {}).get("eps_reported")
    eps_down = bool(eps_fy is not None and eps_prev is not None and eps_fy < eps_prev)
    g_notes = []
    if SCREENER_CONFIG["USE_YIELD_GUARD"] and (cur_yld_fy is None or not (SCREENER_CONFIG["MIN_YIELD"] <= cur_yld_fy <= SCREENER_CONFIG["MAX_YIELD"])):
        g_notes.append(f"배당률범위외({(cur_yld_fy or 0):.1%})")
    if SCREENER_CONFIG["USE_DPS_CAGR_FLOOR"] and (g_dps is None or g_dps < SCREENER_CONFIG["MIN_DPS_CAGR_5Y"]):
        g_notes.append(f"DPS성장<{SCREENER_CONFIG['MIN_DPS_CAGR_5Y']:.0%}")
    if SCREENER_CONFIG["USE_LIQUIDITY_GUARD"] and turnover_60d < SCREENER_CONFIG["MIN_TURNOVER_60D"]:
        g_notes.append(f"저유동성({turnover_60d/1e8:.1f}억)")
    if SCREENER_CONFIG["USE_CRASH_EPS_GUARD"] and dd_52w <= SCREENER_CONFIG["MAX_DD_52W_WITH_EPS_DROP"] and eps_down:
        g_notes.append(f"급락+EPS감소({dd_52w:.1%})")

    return {
        "ticker": d["ticker"], "name": d["name"], "market": d["market"], "path": path, "mcap_eok": d["mcap_eok"],
        "overall": overall, "n_fail": n_fail, "n_warn": n_warn, "n_na": n_na,
        "v1": v1, "v2": v2, "v3": v3, "v4": v4, "v5": v5, "v6": v6, "v7": v7, "v8": v8, "v9": v9,
        "val_basis": val_basis, "val_raw": val_raw, "val_cap": val_cap,
        "dps_g5": g_dps, "eps_g5": g_eps, "tr_3y": tr_3y, "payout_3y": p3,
        "spread_5y": avg5_sp, "close": cur_close, "yield_fy": cur_yld_fy, "yield_val": c_yld,
        "per": c_per, "pbr": c_pbr, "dd_52w": dd_52w, "turnover_60d_eok": turnover_60d / 1e8,
        "guard_pass": len(g_notes) == 0, "guard_note": ", ".join(g_notes) if g_notes else "통과",
    }

JUDGED = [x for d in STAGE2_DATA if (x := judge_screener_one(d, FY)) is not None]
val_col = "val_cap" if SCREENER_CONFIG["USE_VAL_CAP"] else "val_raw"

# 원칙 부합(일반 + 금융자본확인) 우선순위 정렬: 밸류 순위 + DPS 5년 성장 순위 합
TOP_POOL = [x for x in JUDGED if x["overall"].startswith("원칙 부합") and x["guard_pass"] and x[val_col] is not None]
if TOP_POOL:
    vals = [x[val_col] for x in TOP_POOL]
    grs = [x["dps_g5"] if x["dps_g5"] is not None else -99 for x in TOP_POOL]
    for x in TOP_POOL:
        vr = 1 + sum(1 for v in vals if v > x[val_col])
        gr = 1 + sum(1 for g in grs if g > (x["dps_g5"] if x["dps_g5"] is not None else -99))
        x["rank_sum"] = vr + gr
    TOP_POOL.sort(key=lambda x: (x["rank_sum"], -(x[val_col] or -9), -(x["spread_5y"] or -9)))

ALL_DF = pd.DataFrame(JUDGED)
TOP_DF = pd.DataFrame(TOP_POOL)

print(f"■ 스크리닝 완료 요약 (FY{FY}, 기준일 {date.today().isoformat()})")
print(ALL_DF["overall"].value_counts().to_string())
print(f"\n■ 최종 우선순위 후보 (원칙 부합 & 필터 통과): {len(TOP_DF)}종목")

def fmt_table(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return df
    out = pd.DataFrame({
        "종목명": df["name"], "티커": df["ticker"], "시장": df["market"], "경로": df["path"],
        "종합": df["overall"], "가격(9번)": df["v9"],
        "밸류할인": df[val_col].map(lambda v: "-" if pd.isna(v) else f"{v:+.0%}"),
        "DPS 5년": df["dps_g5"].map(lambda v: "-" if pd.isna(v) else f"{v:+.1%}"),
        "EPS 5년": df["eps_g5"].map(lambda v: "-" if pd.isna(v) else f"{v:+.1%}"),
        "배당률(FY)": df["yield_fy"].map(lambda v: "-" if pd.isna(v) else f"{v:.2%}"),
        "총환원율(3년)": df["tr_3y"].map(lambda v: "-" if pd.isna(v) else f"{v:.1f}%"),
        "스프레드": df["spread_5y"].map(lambda v: "-" if pd.isna(v) else f"{v:+.1%}"),
        "52주낙폭": df["dd_52w"].map(lambda v: f"{v:.1%}"),
        "거래대금(억)": df["turnover_60d_eok"].map(lambda v: f"{v:,.1f}"),
        "시총(억)": df["mcap_eok"].map(lambda v: f"{v:,.0f}"),
        "보완필터": df["guard_note"],
    })
    return out

TOP_VIEW = fmt_table(TOP_DF)
ALL_VIEW = fmt_table(ALL_DF.sort_values(["n_fail", "n_warn", "n_na", val_col], ascending=[True, True, True, False]))
display(TOP_VIEW.head(25))

In [ ]:
# ==============================================================================
# 7. 시각화 및 모바일·PC 최적화 단일 HTML 리포트 생성 · 다운로드
# ==============================================================================
as_of_str = date.today().isoformat()

# 2패널 요약 차트 (깔때기 현황 + 원칙 부합 사분면 산점도)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5.5))
fig.suptitle(f"코스피·코스닥 배당성장주 통합 스크리닝 결과 ({as_of_str}, FY{FY})", fontsize=14, fontweight="bold")

# (1) 스크리닝 깔때기
stages = ["탐색 모집단", "1차 배당통과", "2차 판정완료", "원칙 부합"]
counts = [len(uni), len(SURVIVORS), len(JUDGED), len(TOP_DF)]
bars = ax1.bar(stages, counts, color=["#94a3b8", "#60a5fa", "#2563eb", "#059669"], width=0.55)
for b in bars:
    h = b.get_height()
    ax1.text(b.get_x() + b.get_width()/2, h + max(counts)*0.02, f"{int(h):,}", ha="center", va="bottom", fontweight="bold")
ax1.set_title("① 스크리닝 깔때기 (종목 수)", fontsize=11.5, fontweight="bold")
ax1.set_ylabel("종목 수")

# (2) 원칙 부합 종목 사분면: 밸류 할인율 vs DPS 5년 성장률
if not TOP_DF.empty:
    xs = TOP_DF[val_col].clip(-0.6, 1.2) * 100
    ys = TOP_DF["dps_g5"].clip(-0.1, 0.8) * 100
    sizes = (TOP_DF["yield_fy"].fillna(0.02) * 3500).clip(50, 320)
    sc = ax2.scatter(xs, ys, s=sizes, c=TOP_DF["dd_52w"] * 100, cmap="RdYlGn", alpha=0.85, edgecolors="#1f2937", lw=0.7)
    for _, row in TOP_DF.head(15).iterrows():
        xv = max(-60, min(120, (row[val_col] or 0) * 100))
        yv = max(-10, min(80, (row["dps_g5"] or 0) * 100))
        ax2.annotate(row["name"], (xv, yv), xytext=(4, 4), textcoords="offset points", fontsize=8.5)
    ax2.axvline(0, color="gray", ls="--", lw=0.9)
    ax2.axhline(5, color="gray", ls="--", lw=0.9)
    cbar = plt.colorbar(sc, ax=ax2)
    cbar.set_label("52주 고점 대비 낙폭 (%)")
    ax2.set_xlabel("밸류 할인율 (+일수록 과거 10년 중앙값 대비 저평가, %)")
    ax2.set_ylabel("정기 DPS 5년 연평균 성장률 (%)")
    ax2.set_title("② 원칙 부합 후보 맵 (원 크기=배당수익률, 색=52주 낙폭)", fontsize=11.5, fontweight="bold")
else:
    ax2.text(0.5, 0.5, "원칙 부합 종목 없음", ha="center", va="center", fontsize=12)

plt.tight_layout(rect=[0, 0, 1, 0.93])
_buf = io.BytesIO()
fig.savefig(_buf, format="png", dpi=140, bbox_inches="tight", facecolor="white")
CHART_B64 = base64.b64encode(_buf.getvalue()).decode("ascii")
plt.show()

def to_tbl(df: pd.DataFrame) -> str:
    return "<p>해당 종목 없음</p>" if df.empty else df.to_html(index=False, classes="tbl", border=0, escape=True)

html_report = f'''<!doctype html>
<html lang="ko">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>한국 배당성장주 전 종목 스크리닝 리포트 ({as_of_str})</title>
<style>
  :root {{ --bg:#f7f8fa; --card:#fff; --ink:#1f2937; --muted:#6b7280; --border:#e5e7eb; --accent:#0f766e; }}
  * {{ box-sizing:border-box; }}
  body {{ margin:0; padding:16px; background:var(--bg); color:var(--ink); font-family:-apple-system,BlinkMacSystemFont,"NanumGothic","Malgun Gothic",sans-serif; line-height:1.5; }}
  .wrap {{ max-width:1120px; margin:0 auto; }}
  .hero {{ background:linear-gradient(135deg,#064e3b,#0f766e); color:#fff; padding:22px 24px; border-radius:14px; margin-bottom:16px; }}
  .hero h1 {{ margin:0 0 6px; font-size:20px; }}
  .hero p {{ margin:0; opacity:.9; font-size:13px; }}
  .kpis {{ display:grid; grid-template-columns:repeat(auto-fit,minmax(180px,1fr)); gap:10px; margin-bottom:16px; }}
  .kpi {{ background:var(--card); border:1px solid var(--border); border-radius:12px; padding:14px; }}
  .kpi .lab {{ font-size:12px; color:var(--muted); }}
  .kpi .val {{ font-size:22px; font-weight:700; margin-top:4px; }}
  .card {{ background:var(--card); border:1px solid var(--border); border-radius:12px; padding:18px; margin-bottom:16px; }}
  .card h2 {{ margin:0 0 10px; font-size:16px; border-left:4px solid var(--accent); padding-left:10px; }}
  .scroll {{ overflow-x:auto; -webkit-overflow-scrolling:touch; }}
  table.tbl {{ width:100%; border-collapse:collapse; font-size:12.5px; white-space:nowrap; }}
  table.tbl th, table.tbl td {{ padding:8px 10px; border-bottom:1px solid var(--border); text-align:right; }}
  table.tbl th {{ background:#f3f4f6; font-weight:600; }}
  table.tbl th:first-child, table.tbl td:first-child {{ text-align:left; font-weight:600; }}
  img.chart {{ width:100%; height:auto; border-radius:8px; border:1px solid var(--border); }}
  .foot {{ font-size:12px; color:var(--muted); padding:6px 4px; }}
</style>
</head>
<body>
<div class="wrap">
  <div class="hero">
    <h1>🔭 코스피·코스닥 배당성장주 통합 스크리닝 리포트</h1>
    <p>기준일: {as_of_str} · 사업연도: FY{FY} · 대상: {", ".join(SCREENER_CONFIG["MARKETS"])} 시총 {SCREENER_CONFIG["MIN_MCAP_EOK"]:,}억 원 이상 · 활성 보완필터: {", ".join(active_guards) if active_guards else "없음(현행 기준선)"}</p>
  </div>

  <div class="kpis">
    <div class="kpi"><div class="lab">탐색 모집단</div><div class="val">{len(uni):,}종목</div></div>
    <div class="kpi"><div class="lab">1차 배당 이력 통과</div><div class="val">{len(SURVIVORS):,}종목</div></div>
    <div class="kpi"><div class="lab">최종 원칙 부합 (우선순위)</div><div class="val" style="color:#059669">{len(TOP_DF):,}종목</div></div>
    <div class="kpi"><div class="lab">국고채 10년물 (COE)</div><div class="val">{CUR_RF:.2%} ({CUR_RF+0.05:.2%})</div></div>
  </div>

  <div class="card">
    <h2>1. 스크리닝 현황 및 후보 맵</h2>
    <img class="chart" src="data:image/png;base64,{CHART_B64}" alt="스크리닝 차트">
  </div>

  <div class="card">
    <h2>2. 최종 원칙 부합 후보 (재투자 우선순위 정렬 · {len(TOP_VIEW)}종목)</h2>
    <div class="scroll">{to_tbl(TOP_VIEW)}</div>
  </div>

  <div class="card">
    <h2>3. 2차 정밀 판정 전체 목록 ({len(ALL_VIEW)}종목)</h2>
    <div class="scroll">{to_tbl(ALL_VIEW)}</div>
  </div>

  <div class="foot">
    ※ 참고: 금융 경로(은행·보험·금융지주·증권) 종목의 8번 규제자본비율(CET1·K-ICS·NCR)은 사업보고서 본문 공시를 별도로 확인해야 합니다. 본 리포트는 정량 스크리닝 참고 자료이며 투자 권유가 아닙니다.
  </div>
</div>
</body>
</html>'''

report_file = f"screener_report_{as_of_str}.html"
csv_file = f"screener_candidates_{as_of_str}.csv"
Path(report_file).write_text(html_report, encoding="utf-8")
ALL_VIEW.to_csv(csv_file, index=False, encoding="utf-8-sig")

print(f"[리포트 생성 완료] {report_file} ({Path(report_file).stat().st_size/1024:,.1f} KB) · {csv_file}")
if IN_COLAB:
    from google.colab import files
    print("[다운로드 시작] HTML 리포트 파일을 내려받습니다...")
    files.download(report_file)